# Grupo 2 — Detección de deslizamientos con Prithvi
## Versión revisada para Google Colab y ejecución local
**Objetivo:** segmentación binaria de deslizamientos con Landslide4Sense, encoder Prithvi congelado y Random Forest por parches.

**Estado:** ejecutado de extremo a extremo en local el 17 de noviembre de 2026. Resultados reales en `taller_deslizamientos/resultados/` y `entregables/`.

**Hallazgo principal:** en el subconjunto asignado (4 train / 2 test) la clase deslizamiento no aparece en las etiquetas de parche por la regla de mayoría del 50 por ciento. El modelo empata con la línea base (IoU y F1 de deslizamiento iguales a 0).

**Antes de interpretar:** confirmar con el profesor la correspondencia de los 14 canales Landslide4Sense y los seis canales seleccionados.


## 0. Portada, contribuciones y pregunta de investigación
**Curso:** Inteligencia Artificial  
**Profesor:** Eduardo Enrique Zurek Varela  
**Grupo:** 2 — Detección de deslizamientos  
**Fecha de ejecución verificada:** 17 de noviembre de 2026  
**Repositorio:** https://github.com/Charlsz/prithvi-deslizamiento

**Integrantes y contribuciones**
- Bauz Villa, Juan: revisión de la ficha del grupo, formulación del problema y apoyo en el marco teórico.
- Cera Moran, Daniel: documentación de datos, canales, leyenda de clases y partición train/test.
- Galvis Pajaro, Carlos: ejecución del experimento, corrección de carga de Prithvi, organización del repositorio y verificación de entregables.
- Jimenez Atencia, Cher: apoyo en implementación del notebook, registro de procedencia y versiones.
- Mercado Martinez, Sebastian: evaluación cuantitativa, tablas de métricas y comparación con la línea base.
- Molina Vega, Dayana: análisis de errores, revisión de figuras y redacción de limitaciones.
- Romero Pemberthy, Juan: propuesta de transferencia a Colombia y criterios de validación regional.
- Ruiz Espitia, Kevin: consideraciones éticas, riesgos de uso y declaración de integridad.
- Vargas Becerra, Natalia: conclusiones, referencias en formato IEEE y revisión final del informe.

**Pregunta:** ¿Qué tan bien identifica un encoder congelado Prithvi con un clasificador Random Forest las regiones etiquetadas como deslizamiento en el subconjunto de evaluación?

**Hipótesis (puesta a prueba):** las características del encoder permiten superar una línea base constante para la clase positiva.

**Resultado empírico del piloto:** la hipótesis no se sostiene en este subconjunto. El Random Forest no vio ningún parche de entrenamiento etiquetado como deslizamiento, porque la agregación mayoritaria 16×16 exige al menos 50 por ciento de píxeles positivos y el máximo observado fue cerca de 40 por ciento. Por eso el modelo predijo siempre fondo, igual que la línea base.

**Declaración de uso de IA:** se usó asistencia de IA para organizar el notebook, corregir la lectura del YAML de Prithvi, ejecutar el piloto y redactar el informe. Las métricas, figuras y archivos CSV se generaron con una ejecución real; los valores no son inventados.


## Marco teórico y formulación
**Modelo fundacional geoespacial:** red preentrenada sobre imágenes de observación terrestre para aprender representaciones transferibles. **Transformer:** procesa tokens de parches y usa atención para relacionar sus características. **MAE:** durante el preentrenamiento aprende a reconstruir regiones ocultas; esto **no equivale** a predecir una máscara de deslizamientos. **Inferencia:** ejecutar el modelo con datos nuevos. **Fine-tuning:** actualizar pesos preentrenados para una tarea; aquí **no** se realiza sobre Prithvi. **Cabeza de predicción:** Random Forest entrenado con características del encoder congelado.

**Entrada:** recortes satelitales multibanda leídos por el módulo docente, transformados a 6 canales ópticos; verificar orden, escala y producto contra metadatos del dataset. **Salida:** máscara binaria de 224×224 píxeles (0=fondo, 1=deslizamiento); las posiciones ignoradas no cuentan. **Unidad de predicción:** parche de 16×16, expandido a píxeles para evaluación. Esta segmentación por parches tiene resolución espacial limitada.

**Nota de alcance:** el experimento clasifica evidencia visible en imágenes; no predice futuros deslizamientos. El pequeño subconjunto de 4 muestras de entrenamiento y 2 de prueba no permite inferir rendimiento operativo general.

In [ ]:
%pip -q install "timm==0.9.2" "huggingface_hub>=0.25,<1" "rasterio>=1.3,<2" "PyYAML>=6,<7" "einops>=0.7,<1" "h5py>=3,<4" "scikit-learn>=1.3,<2" requests pandas matplotlib


## 1. Módulo de soporte incluido
Esta celda escribe el módulo que acompaña al paquete. El notebook es autónomo: no necesita buscar o subir un archivo `.py` adicional. El código remoto del modelo oficial se fija a una revisión y se importa más adelante; el docente debe revisarlo en el piloto.


In [ ]:
from pathlib import Path
MODULE_TEXT = '"""Soporte docente experimental para Prithvi. No incluye pesos ni escenas reales.\nLas descargas se realizan en Colab. No validado de extremo a extremo.\n"""\nfrom pathlib import Path\nimport json, re, hashlib, tarfile, shutil, warnings, io, time\nimport numpy as np\nimport requests\n\nSOURCES = {\n \'inundaciones\': \'https://github.com/cloudtostreet/Sen1Floods11\',\n \'quemadas\': \'https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars\',\n \'cultivos\': \'https://huggingface.co/datasets/ibm-nasa-geospatial/multi-temporal-crop-classification\',\n \'deslizamientos\': \'https://huggingface.co/datasets/sasmit545/Landslide4sense\',\n}\n\ndef save_json(path, data):\n    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False))\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef get(url,path,max_mb=250):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    if path.exists() and path.stat().st_size>0:return path\n    temp=path.with_suffix(path.suffix+\'.part\')\n    try:\n        with requests.get(url,stream=True,timeout=(30,120)) as r:\n            r.raise_for_status(); size=0\n            with open(temp,\'wb\') as f:\n                for b in r.iter_content(1024*1024):\n                    size+=len(b)\n                    if size>max_mb*1024**2:raise RuntimeError(\'Descarga supera el límite por archivo.\')\n                    f.write(b)\n        temp.replace(path)\n    finally:\n        if temp.exists():temp.unlink()\n    return path\n\ndef sample_key(path):\n    stem=Path(path).name.lower()\n    for suffix in [\'.tiff\',\'.tif\',\'.h5\',\'.hdf5\']:\n        if stem.endswith(suffix):stem=stem[:-len(suffix)]\n    for token in [\'_merged\',\'.mask\',\'_mask\',\'_label\',\'_s2hand\',\'_labelhand\']:\n        stem=stem.replace(token,\'\')\n    return stem\n\ndef pair_tifs(directory):\n    files=sorted(Path(directory).rglob(\'*.tif\'))\n    labels={sample_key(f):f for f in files if any(s in f.name.lower() for s in [\'mask\',\'label\'])}\n    return [(f,labels[sample_key(f)]) for f in files\n            if not any(s in f.name.lower() for s in [\'mask\',\'label\']) and sample_key(f) in labels]\n\nclass LimitedReader:\n    def __init__(self, raw, max_bytes):self.raw=raw; self.maximum=max_bytes; self.total=0\n    def read(self,size=-1):\n        b=self.raw.read(size); self.total+=len(b)\n        if self.total>self.maximum:\n            raise RuntimeError(\'El tar excede el presupuesto de red. No se completó el subconjunto. Aumentar max_stream_mb solo tras revisar tamaño y conexión; no hay garantía de descarga pequeña.\')\n        return b\n\ndef subset_archive(url, root, requested, combined=False, max_stream_mb=512):\n    """Extrae pares sin descomprimir rutas del tar. Conserva pocos archivos.\n    Leer gzip es secuencial: puede transferir mucho aunque se guarde poco.\n    """\n    root=Path(root); root.mkdir(parents=True,exist_ok=True)\n    targets={\'train\':requested[0],\'test\':requested[1]} if combined else requested\n    kept={s:{\'image\':set(),\'label\':set()} for s in targets}\n    with requests.get(url,stream=True,timeout=(30,120)) as response:\n        response.raise_for_status(); response.raw.decode_content=True\n        counter=LimitedReader(response.raw,int(max_stream_mb*1024**2))\n        with tarfile.open(fileobj=counter,mode=\'r|gz\') as tar:\n            for member in tar:\n                if not member.isfile() or not member.name.lower().endswith((\'.tif\',\'.tiff\')):continue\n                low=member.name.lower()\n                if combined:\n                    if re.search(r\'(^|[/_])(?:training|train)([/_]|$)\',low): split=\'train\'\n                    elif re.search(r\'(^|[/_])(?:validation|valid|val|test)([/_]|$)\',low): split=\'test\'\n                    else:continue\n                else:split=next(iter(targets))\n                target=targets[split]\n                if len(pair_tifs(root/split))>=target:continue\n                kind=\'label\' if any(t in Path(low).name for t in [\'mask\',\'label\']) else \'image\'\n                key=sample_key(low)\n                other=\'image\' if kind==\'label\' else \'label\'\n                # Guardado acotado; si se listan antes imágenes que máscaras,\n                # conservar hasta 4*n para permitir el emparejamiento.\n                if key not in kept[split][other] and len(kept[split][kind])>=4*target:continue\n                dest=root/split/Path(member.name).name; dest.parent.mkdir(exist_ok=True)\n                stream=tar.extractfile(member)\n                with open(dest,\'wb\') as out:shutil.copyfileobj(stream,out)\n                kept[split][kind].add(key)\n                if all(len(pair_tifs(root/s))>=n for s,n in targets.items()):break\n        print(\'Bytes comprimidos leídos:\',counter.total)\n    result={s:pair_tifs(root/s)[:n] for s,n in targets.items()}\n    for s,n in targets.items():\n        if len(result[s])<n:\n            raise RuntimeError(f\'Solo se emparejaron {len(result[s])}/{n} en {s}. Revisar nombres/estructura del tar; no se sustituyen datos.\')\n    return result\n\ndef download_flood(root,ntrain,ntest):\n    import pandas as pd\n    root=Path(root); base=\'https://storage.googleapis.com/sen1floods11/v1.1/\'\n    result={}\n    for split,tag,n in [(\'train\',\'train\',ntrain),(\'test\',\'test\',ntest)]:\n        csvname=f\'flood_{tag}_data.csv\'\n        alternatives=[base+\'splits/flood_handlabeled/\'+csvname,\n          \'https://raw.githubusercontent.com/cloudtostreet/Sen1Floods11/master/splits/flood_handlabeled/\'+csvname]\n        content=None\n        for url in alternatives:\n            rr=requests.get(url,timeout=60)\n            if rr.status_code==200:content=rr.text;break\n        if content is None:raise RuntimeError(\'No se pudo recuperar la partición oficial Sen1Floods11.\')\n        (root).mkdir(parents=True,exist_ok=True); (root/csvname).write_text(content)\n        ids=[]\n        for line in content.splitlines():\n            hit=re.search(r\'([A-Za-z]+_\\d+)_S[12]Hand\\.tif\',line)\n            if hit and hit.group(1) not in ids:ids.append(hit.group(1))\n        if len(ids)<n:raise RuntimeError(\'La estructura del CSV cambió: revisar antes de continuar.\')\n        rows=[]\n        for sample in ids[:n]:\n            image=get(base+\'data/flood_events/HandLabeled/S2Hand/\'+sample+\'_S2Hand.tif\',root/split/(sample+\'_S2Hand.tif\'))\n            label=get(base+\'data/flood_events/HandLabeled/LabelHand/\'+sample+\'_LabelHand.tif\',root/split/(sample+\'_LabelHand.tif\'))\n            rows.append((image,label))\n        result[split]=rows\n    return result\n\ndef download_landslide(root,ntrain,ntest):\n    from huggingface_hub import HfApi,hf_hub_download\n    repo=\'sasmit545/Landslide4sense\'; api=HfApi()\n    rev=api.dataset_info(repo).sha\n    names=api.list_repo_files(repo,repo_type=\'dataset\',revision=rev)\n    result={}\n    for split,n in [(\'train\',ntrain),(\'test\',ntest)]:\n        images=sorted([f for f in names if f.startswith(\'images/\'+split+\'/\') and f.endswith(\'.h5\')])\n        masks={re.findall(r\'\\d+\',Path(f).stem)[-1]:f for f in names\n               if f.startswith(\'annotations/\'+split+\'/\') and f.endswith(\'.h5\')}\n        rows=[]\n        for im in images:\n            idx=re.findall(r\'\\d+\',Path(im).stem)[-1]\n            if idx not in masks:continue\n            a=hf_hub_download(repo,filename=im,repo_type=\'dataset\',revision=rev,local_dir=str(root))\n            b=hf_hub_download(repo,filename=masks[idx],repo_type=\'dataset\',revision=rev,local_dir=str(root))\n            rows.append((Path(a),Path(b)))\n            if len(rows)==n:break\n        if len(rows)<n:raise RuntimeError(\'Estructura HDF5 diferente de la documentada; revisar el inventario.\')\n        result[split]=rows\n    save_json(Path(root)/\'revision_datos.json\',{\'repo\':repo,\'revision\':rev})\n    return result\n\ndef download_data(task,root,ntrain=4,ntest=2,max_stream_mb=512):\n    from huggingface_hub import HfApi,hf_hub_url\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    if task==\'inundaciones\': result=download_flood(root,ntrain,ntest)\n    elif task==\'deslizamientos\':result=download_landslide(root,ntrain,ntest)\n    else:\n        repo={\'quemadas\':\'ibm-nasa-geospatial/hls_burn_scars\',\n              \'cultivos\':\'ibm-nasa-geospatial/multi-temporal-crop-classification\'}[task]\n        api=HfApi();revision=api.dataset_info(repo).sha\n        files=api.list_repo_files(repo,repo_type=\'dataset\',revision=revision)\n        print(\'Inventario del dataset:\',files)\n        if task==\'quemadas\':\n            archive=next((f for f in files if f.endswith(\'hls_burn_scars.tar.gz\')),None)\n            if archive is None:raise RuntimeError(\'No se encontró el tar documentado de Burn Scars.\')\n            result=subset_archive(hf_hub_url(repo,archive,repo_type=\'dataset\',revision=revision),root,\n                                  (ntrain,ntest),combined=True,max_stream_mb=max_stream_mb)\n        else:\n            result={}\n            for split,name,n in [(\'train\',\'training_chips.tgz\',ntrain),(\'test\',\'validation_chips.tgz\',ntest)]:\n                if name not in files:raise RuntimeError(f\'Falta {name}; revisar inventario.\')\n                result.update(subset_archive(hf_hub_url(repo,name,repo_type=\'dataset\',revision=revision),root,\n                         {split:n},combined=False,max_stream_mb=max_stream_mb))\n        save_json(root/\'revision_datos.json\',{\'repo\':repo,\'revision\':revision})\n    manifest=[]\n    for split,pairs in result.items():\n        for im,lab in pairs:\n            manifest.append({\'split\':split,\'image\':str(im),\'label\':str(lab),\n                             \'image_sha256\':sha256(im),\'label_sha256\':sha256(lab)})\n    save_json(root/\'manifest.json\',{\'task\':task,\'source\':SOURCES[task],\n          \'note\':\'Subconjunto por orden, no por desempeño; train/test según proveedor. En cultivos, validation se usa como evaluación sin ajustar decisiones.\',\n          \'samples\':manifest})\n    return result\n\ndef read_sample(task, image_path, label_path, crop_layout=\'CT\',side=224):\n    """Retorna imagen C,T,H,W y etiqueta H,W. No remuestrea etiquetas.\n    Orden óptico asumido/documentado en el notebook: azul,verde,rojo,B8A,SWIR1,SWIR2.\n    """\n    import rasterio,h5py\n    if task==\'deslizamientos\':\n        with h5py.File(image_path,\'r\') as f:\n            keys=list(f.keys());key=\'img\' if \'img\' in f else \'image\' if \'image\' in f else None\n            if key is None:raise RuntimeError(f\'Clave HDF5 no reconocida: {keys}\')\n            im=np.asarray(f[key],dtype=np.float32)\n        if im.shape[-1]==14:im=im.transpose(2,0,1)\n        if im.shape[0]!=14:raise ValueError(\'Se esperaban 14 canales Landslide4Sense.\')\n        im=im[[1,2,3,8,10,11]][:,None] # B02 B03 B04 B8A B11 B12; excluye relieve\n        with h5py.File(label_path,\'r\') as f:\n            if \'mask\' not in f:raise RuntimeError(\'No existe la clave mask en la referencia.\')\n            label=np.asarray(f[\'mask\']).squeeze().astype(np.int64)\n    else:\n        with rasterio.open(image_path) as f:\n            arr=f.read(masked=True).astype(np.float32).filled(np.nan)\n            ref_crs,ref_transform=f.crs,f.transform\n        with rasterio.open(label_path) as f:\n            label=f.read(1,masked=True).astype(np.int64).filled(-1)\n            if f.crs!=ref_crs or f.transform!=ref_transform:raise ValueError(\'Imagen y etiqueta geográficamente desalineadas.\')\n        if task==\'inundaciones\':\n            if arr.shape[0]!=13:raise ValueError(\'Se esperan las 13 bandas S2Hand en orden B01..B12, incluyendo B8A.\')\n            im=arr[[1,2,3,8,11,12]][:,None]\n        elif task==\'quemadas\':\n            if arr.shape[0]!=6:raise ValueError(\'Se esperan 6 bandas HLS.\')\n            im=arr[:,None]\n        else:\n            if arr.shape[0]!=18:raise ValueError(\'Se esperan 18 canales (6 bandas × 3 fechas).\')\n            if crop_layout==\'CT\':im=arr.reshape(6,3,*arr.shape[1:])\n            elif crop_layout==\'TC\':im=arr.reshape(3,6,*arr.shape[1:]).transpose(1,0,2,3)\n            else:raise ValueError(\'crop_layout debe ser CT o TC; confirmar la documentación de la versión.\')\n            label=np.where(label==0,-1,label) # código 0 = nodata; no desplaza clases 1..13\n    if im.shape[-2:]!=label.shape:raise ValueError(\'Tamaño de etiqueta incompatible.\')\n    valid=np.isfinite(im).all(axis=(0,1)) & (im!=-9999).all(axis=(0,1))\n    label=np.where(valid,label,-1)\n    h,w=label.shape; r=max(0,(h-side)//2);c=max(0,(w-side)//2)\n    im=im[:,:,r:r+side,c:c+side];label=label[r:r+side,c:c+side]\n    ph=side-im.shape[-2];pw=side-im.shape[-1]\n    # Bordes de imágenes menores se repiten para contexto; la etiqueta de relleno es ignorada.\n    if ph or pw:\n        im=np.pad(im,((0,0),(0,0),(0,ph),(0,pw)),mode=\'edge\')\n        label=np.pad(label,((0,ph),(0,pw)),constant_values=-1)\n    return im.astype(\'float32\'),label\n\ndef train_normalization(samples):\n    # Estadísticas de entrenamiento exclusivamente, por banda y todas sus fechas.\n    vals=[np.where(y[None,None]>=0,x,np.nan).reshape(6,-1) for x,y in samples]\n    joined=np.concatenate(vals,axis=1)\n    mean=np.nanmean(joined,axis=1); std=np.nanstd(joined,axis=1)\n    if not np.isfinite(mean).all():raise ValueError(\'No hay datos válidos para normalizar.\')\n    return mean,np.maximum(std,1e-6)\n\ndef normalize(x,mean,std):\n    x=(x-mean[:,None,None,None])/std[:,None,None,None]\n    return np.nan_to_num(x,nan=0,posinf=0,neginf=0).astype(\'float32\')\n\ndef load_prithvi(root,num_frames=1):\n    import torch,yaml,sys,importlib.util\n    from huggingface_hub import HfApi,hf_hub_download\n    repo=\'ibm-nasa-geospatial/Prithvi-EO-1.0-100M\'\n    revision=\'7b231fe5b0684e2ddd40c967fbc02e225ef77dc6\'\n    names=HfApi().list_repo_files(repo,revision=revision)\n    def choose(options):\n        for f in options:\n            if f in names:return f\n        raise RuntimeError(\'No está el archivo esperado en la revisión fijada: \'+str(options))\n    py=choose([\'Prithvi.py\',\'prithvi.py\',\'prithvi_mae.py\'])\n    cf=choose([\'Prithvi_100M_config.yaml\',\'config.yaml\'])\n    wt=choose([\'Prithvi_100M.pt\'])\n    def dl(f):return Path(hf_hub_download(repo,filename=f,revision=revision,local_dir=str(root)))\n    py=dl(py); cf=dl(cf); wt=dl(wt)\n    # Ejecuta Python remoto del repositorio oficial, fijado a revisión. Revisar antes de distribuir.\n    spec=importlib.util.spec_from_file_location(\'prithvi_official\',py)\n    mod=importlib.util.module_from_spec(spec);sys.path.insert(0,str(py.parent));spec.loader.exec_module(mod)\n    klass=next((getattr(mod,n) for n in [\'MaskedAutoencoderViT\',\'PrithviMAE\'] if hasattr(mod,n)),None)\n    if klass is None:raise RuntimeError(\'Clase MAE no encontrada.\')\n    cfg=yaml.safe_load(cf.read_text())\n    if isinstance(cfg, dict) and \'model_args\' in cfg:\n        args=dict(cfg[\'model_args\'])\n    else:\n        allowed={\'img_size\',\'patch_size\',\'num_frames\',\'tubelet_size\',\'in_chans\',\'embed_dim\',\'depth\',\'num_heads\',\'decoder_embed_dim\',\'decoder_depth\',\'decoder_num_heads\',\'mlp_ratio\',\'norm_pix_loss\'}\n        args={k:v for k,v in cfg.items() if k in allowed}\n        if \'bands\' in cfg and \'in_chans\' not in args:\n            args[\'in_chans\']=len(cfg[\'bands\'])\n    args[\'num_frames\']=num_frames\n    if \'in_chans\' not in args:\n        args[\'in_chans\']=6\n    model=klass(**args)\n    state=torch.load(wt,map_location=\'cpu\',weights_only=True)\n    if \'state_dict\' in state:state=state[\'state_dict\']\n    if \'model\' in state and isinstance(state[\'model\'],dict):state=state[\'model\']\n    state=dict(state)\n    for key in [\'pos_embed\',\'decoder_pos_embed\']:state.pop(key,None)\n    status=model.load_state_dict(state,strict=False)\n    if set(status.missing_keys)-{\'pos_embed\',\'decoder_pos_embed\'} or status.unexpected_keys:\n        raise RuntimeError(\'Checkpoint incompatible: \'+str(status))\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    model=model.to(device).eval()\n    for param in model.parameters():param.requires_grad=False\n    return model,device,{\'repo\':repo,\'revision\':revision,\'checkpoint\':str(wt),\'device\':device,\'num_frames\':num_frames}\n\ndef features(model,device,image):\n    import torch\n    x=torch.from_numpy(image.copy())[None].to(device)\n    torch.manual_seed(0)\n    with torch.inference_mode():\n        latent,mask,restore=model.forward_encoder(x,mask_ratio=0.)\n        tokens=latent[:,1:,:]\n        # Aunque mask_ratio=0, MAE puede permutar los parches. Recuperar orden espacial.\n        tokens=torch.gather(tokens,1,restore.unsqueeze(-1).expand(-1,-1,tokens.shape[-1]))\n    T=image.shape[1];D=tokens.shape[-1]\n    if tokens.shape[1]!=T*14*14:raise RuntimeError(\'Longitud de tokens distinta de la esperada (224, parches16).\')\n    return tokens.reshape(T,14,14,D).mean(dim=0).cpu().numpy()\n\ndef patch_labels(label,size=16,min_fraction=.5):\n    result=np.full((14,14),-1,dtype=np.int64)\n    for r in range(14):\n        for c in range(14):\n            block=label[r*size:(r+1)*size,c*size:(c+1)*size]\n            v=block[block>=0]\n            if v.size>=min_fraction*size*size:\n                classes,counts=np.unique(v,return_counts=True)\n                result[r,c]=classes[np.argmax(counts)]\n    return result\n\ndef confusion(y,pred,classes):\n    y=np.asarray(y); pred=np.asarray(pred)\n    if y.shape!=pred.shape:raise ValueError(\'Formas distintas.\')\n    valid=y>=0; a=y[valid];b=pred[valid]\n    if not valid.any():raise ValueError(\'No hay píxeles válidos.\')\n    if not np.isin(a,classes).all() or not np.isin(b,classes).all():raise ValueError(\'Clase fuera de la leyenda.\')\n    mat=np.zeros((len(classes),len(classes)),dtype=np.int64)\n    for i,c in enumerate(classes):\n        for j,d in enumerate(classes):mat[i,j]=np.count_nonzero((a==c)&(b==d))\n    return mat\n\ndef metrics(mat,classes):\n    rows=[]\n    for i,cl in enumerate(classes):\n        tp=int(mat[i,i]); fp=int(mat[:,i].sum()-tp); fn=int(mat[i,:].sum()-tp)\n        support=int(mat[i,:].sum());union=tp+fp+fn\n        iou=tp/union if union else np.nan\n        f1=2*tp/(2*tp+fp+fn) if union else np.nan\n        precision=tp/(tp+fp) if tp+fp else (0. if support else np.nan)\n        recall=tp/(tp+fn) if support else np.nan\n        rows.append({\'class\':int(cl),\'support\':support,\'iou\':iou,\'f1\':f1,\'precision\':precision,\'recall\':recall})\n    return rows\n\ndef smoke_test():\n    y=np.array([[0,0],[1,-1]]);p=np.array([[0,1],[1,0]])\n    mat=confusion(y,p,[0,1]);assert np.array_equal(mat,[[1,1],[0,1]])\n    row=metrics(mat,[0,1])[1]\n    assert np.isclose(row[\'iou\'],.5) and np.isclose(row[\'f1\'],2/3)\n    # nodata debe excluirse; nunca se cuenta como fondo.\n    assert mat.sum()==3\n    return \'Tests de métricas superados; NO prueban descarga ni inferencia.\'\n'
MODULE_FILE = (Path('/content') if Path('/content').exists() else Path('.').resolve()) / 'prithvi_taller.py'
MODULE_FILE.write_text(MODULE_TEXT, encoding='utf-8')
import sys
sys.path.insert(0, str(MODULE_FILE.parent))
from prithvi_taller import *
print(smoke_test())
print('Módulo escrito en:', MODULE_FILE)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.ensemble import RandomForestClassifier
from importlib.metadata import version

TASK='deslizamientos'
CLASSES=[0, 1]
# Compatible con Google Colab y con ejecución local en Windows/Linux/Mac.
BASE = Path('/content') if Path('/content').exists() else Path('.').resolve()
ROOT = BASE / ('taller_' + TASK)
ROOT.mkdir(exist_ok=True)
OUT = ROOT / 'resultados'
OUT.mkdir(exist_ok=True)
NTRAIN=4
NTEST=2
MAX_STREAM_MB=512  # Por archivo comprimido; no es límite total. Puede detener la descarga.
CROP_LAYOUT='CT'  # Solo aplica a cultivos; se conserva por compatibilidad con el módulo docente.
print('Tarea:', TASK, '| CPU/GPU:', 'GPU' if torch.cuda.is_available() else 'CPU')
print('Raíz de trabajo:', ROOT)
print('Clases:', CLASSES)


## 2. Descargar imágenes y etiquetas reales
Fuente: https://huggingface.co/datasets/sasmit545/Landslide4sense

Los archivos se descargan a `/content/taller_deslizamientos/datos`. Se conserva un manifiesto con nombres, particiones y hashes. Solo se seleccionan archivos por su orden en la fuente, sin evaluar primero sus resultados.

**Leyenda:** 0: no deslizamiento; 1: deslizamiento observado. No predice eventos futuros.

Los datasets no están garantizados como globalmente independientes por evento. Se conserva la separación disponible del proveedor; deben estudiarse sus límites.

Si la descarga se detiene por presupuesto de red o por cambios de estructura, **detenga el experimento** y registre el mensaje; no invente archivos ni reemplace la práctica por los tests artificiales. El docente debe validar el acceso antes de asignar este material.


In [ ]:
pairs=download_data(TASK,ROOT/'datos',ntrain=NTRAIN,ntest=NTEST,max_stream_mb=MAX_STREAM_MB)
for split,rows in pairs.items():
    print(split,[(im.name,lab.name) for im,lab in rows])
print('Manifiesto:',ROOT/'datos'/'manifest.json')


## 3. Leer, verificar y explorar
Se usan seis bandas ópticas equivalentes en significado: azul, verde, rojo, NIR estrecho (B8A), SWIR1 y SWIR2. No se mezclan pendiente o elevación con bandas ópticas.

- Inundaciones: S2Hand original de 13 bandas → índices base cero `[1,2,3,8,11,12]`.
- Áreas quemadas: producto HLS de seis bandas en el orden indicado por su documentación.
- Cultivos: 18 canales → seis bandas × tres fechas. La disposición `CT` debe cotejarse con la documentación/lector de la versión; usar `TC` solo si la fuente realmente es fecha–banda.
- Deslizamientos: 14 canales → se seleccionan seis ópticos `[1,2,3,8,10,11]` según la convención de 12 bandas S2 sin B10, más dos auxiliares. **La correspondencia debe verificarse antes del piloto**. No se utilizan las variables de relieve.

Cada escena produce un recorte central 224×224. Si la entrada es menor, se rellena contexto y se ignoran las etiquetas de relleno. No se remuestrean etiquetas.

Esta adaptación no equivale a los preprocesamientos de los checkpoints especializados. Las diferencias de sensor, escala y procesamiento son límites importantes del estudio.


In [ ]:
samples={split:[read_sample(TASK,im,lab,crop_layout=CROP_LAYOUT) for im,lab in rows]
         for split,rows in pairs.items()}
for split,rows in samples.items():
    print(split,[(x.shape,y.shape,np.unique(y).tolist()) for x,y in rows])
    for x,y in rows:
        if not np.isin(y[y>=0],CLASSES).all(): raise ValueError('Etiqueta fuera de leyenda; revisar la versión.')
x0,y0=samples['train'][0]

def display_rgb(x, label=None):
    a=x[[2,1,0],0].transpose(1,2,0).copy()
    for c in range(3):
        v=a[:,:,c]; finite=np.isfinite(v)
        if label is not None:
            finite = finite & (label>=0)
        if not finite.any():
            continue
        lo,hi=np.percentile(v[finite],[2,98]); a[:,:,c]=np.clip((v-lo)/max(hi-lo,1e-6),0,1)
    a=np.nan_to_num(a)
    if label is not None:
        a[label<0]=0.15
    return a

fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].imshow(display_rgb(x0,y0));ax[0].set_title('RGB de entrenamiento (zona válida)')
ax[1].imshow(np.ma.masked_where(y0<0,y0),vmin=min(CLASSES),vmax=max(CLASSES),cmap='tab20');ax[1].set_title('Referencia; ignorados ocultos')
for a in ax:a.axis('off')
fig.tight_layout();fig.savefig(OUT/'01_exploracion.png',dpi=150);plt.show()
print('Registro de exploración: 6 canales ópticos, recorte/pad a 224x224, leyenda 0/1, inválidos=-1.')
print('Las imágenes Landslide4Sense suelen ser 128x128; el relleno edge completa a 224 y se ignora en la evaluación.')


**Actividad:** registre canales, formas, leyenda, datos faltantes y correspondencia con la documentación. Explique qué información se pierde al usar solo el recorte central y qué podría cambiar al utilizar otras fechas o variables auxiliares.


## 4. Normalización sin fuga de prueba
Se estima media y desviación por banda **solo con imágenes de entrenamiento**, usando posiciones con etiqueta válida, sin usar su clase. Estas estadísticas se aplican a ambos conjuntos. Esto no es fine-tuning ni la normalización oficial original: es una decisión de transferencia que permite trabajar con distintas escalas, y debe mencionarse como limitación.

Imputación: los valores no finitos se convierten en cero después de estandarizar. No se evalúan posiciones inválidas.


In [ ]:
mean,std=train_normalization(samples['train'])
prepared={split:[(normalize(x,mean,std),y) for x,y in rows] for split,rows in samples.items()}
save_json(OUT/'normalizacion.json',{'method':'train_zscore_by_band','mean':mean.tolist(),'std':std.tolist()})
print('Media train:',mean,'| STD train:',std)


## 5. Descargar Prithvi y congelar el encoder
Se fija una revisión histórica del modelo EO-1.0-100M. Se regenera la codificación posicional fija cuando se cambia el número de fechas. Solo esas claves pueden faltar al cargar los pesos; cualquier incompatibilidad restante detiene el proceso.

El checkpoint se carga con `weights_only=True`; no habilite carga arbitraria si aparece un error. El forward del encoder no actualiza pesos. Los tokens espaciales se restauran al orden original porque MAE puede permutarlos incluso sin ocultar parches.


In [ ]:
T=prepared['train'][0][0].shape[1]
model,device,model_info=load_prithvi(ROOT/'modelo',num_frames=T)
provenance={'task':TASK,'model':model_info,'ntrain':NTRAIN,'ntest':NTEST,
 'classes':CLASSES,'crop_layout':CROP_LAYOUT,'crop':'center224_pad_ignore',
 'normalization':'train_zscore_by_band','head':'RandomForest, patch16 majority, frozen encoder',
 'packages':{p:version(p) for p in ['torch','torchvision','timm','numpy','scikit-learn','rasterio']}}
save_json(OUT/'procedencia.json',provenance)
print(model_info)


### Advertencia metodológica
El subconjunto por defecto (4 escenas de entrenamiento y 2 de prueba) es un **piloto pequeño**, no evidencia de generalización regional. La prueba se mantiene separada. Si el profesor asignó más escenas, cambiar `NTRAIN` y `NTEST` antes de la descarga, sin seleccionar muestras según resultados.


## 6. Entrenar exclusivamente la cabeza ligera
Cada parche 16×16 recibe la clase mayoritaria si ≥50% de sus etiquetas son válidas. Los empates se resuelven por el primer código en orden ascendente. El Random Forest opera sobre los embeddings de Prithvi. En cultivos se promedian los tokens de las fechas **después de la atención espaciotemporal**.

No se cambia ningún peso del encoder. Se conserva el conjunto de evaluación sin usarlo para elegir hiperparámetros. Si el subconjunto de entrenamiento no contiene una clase, el clasificador no puede aprenderla; se informa el problema, no se buscan escenas para mejorar la nota.


In [ ]:
train_features=[];train_targets=[]
for i,(x,y) in enumerate(prepared['train']):
    f=features(model,device,x);label=patch_labels(y)
    valid=label>=0
    train_features.append(f[valid]);train_targets.append(label[valid])
    print('Entrenamiento escena',i,'parches válidos:',valid.sum())
X=np.concatenate(train_features);Y=np.concatenate(train_targets)
if len(Y)==0:raise ValueError('No hay parches de entrenamiento válidos.')
print('Distribución de etiquetas por parche:',np.unique(Y,return_counts=True))
missing=sorted(set(CLASSES)-set(np.unique(Y)))
if missing:print('LIMITACIÓN: clases ausentes del entrenamiento:',missing)
head=RandomForestClassifier(n_estimators=80,max_depth=10,class_weight='balanced',random_state=42,n_jobs=2)
assert np.isfinite(X).all(), "Embeddings contienen NaN o infinito"
head.fit(X,Y)
# Baseline: clase de píxel más frecuente SOLO en entrenamiento.
train_labels=np.concatenate([y[y>=0] for _,y in samples['train']])
values,counts=np.unique(train_labels,return_counts=True)
majority=int(values[np.argmax(counts)])
print('Línea base constante:',majority)


## 7. Evaluar el modelo y la línea base
La clase predicha por parche se expande a 16×16 mediante repetición. Es una máscara gruesa; no implica resolución efectiva de un píxel. Se compara con la referencia original sobre posiciones válidas.

Se acumulan matrices de confusión globales por píxel; además se guardan métricas por escena. No se interpreta el conjunto de píxeles como observaciones geográficas independientes. Las clases ausentes de referencia y predicción tienen métrica no definida y se excluyen del promedio macro; las clases presentes con omisiones sí se penalizan.


In [ ]:
totals={'prithvi_rf':np.zeros((len(CLASSES),len(CLASSES)),dtype=np.int64),
        'baseline':np.zeros((len(CLASSES),len(CLASSES)),dtype=np.int64)}
scene_rows=[]
for i,((x,y),(raw,_)) in enumerate(zip(prepared['test'],samples['test'])):
    f=features(model,device,x)
    grid=head.predict(f.reshape(-1,f.shape[-1])).reshape(f.shape[0],f.shape[1])
    pred=np.repeat(np.repeat(grid,16,axis=0),16,axis=1)
    baseline=np.full(y.shape,majority,dtype=np.int64)
    for name,p in [('prithvi_rf',pred),('baseline',baseline)]:
        mat=confusion(y,p,CLASSES);totals[name]+=mat
        for row in metrics(mat,CLASSES):scene_rows.append({'scene':i,'method':name,**row})
    np.savez_compressed(OUT/f'escena_{i}.npz',label=y,prediction=pred,baseline=baseline)
    fig,ax=plt.subplots(1,4,figsize=(15,4))
    ax[0].imshow(display_rgb(raw,y));ax[0].set_title('RGB fecha 1 (zona válida)')
    ax[1].imshow(np.ma.masked_where(y<0,y),vmin=min(CLASSES),vmax=max(CLASSES),cmap='tab20');ax[1].set_title('Referencia')
    ax[2].imshow(np.ma.masked_where(y<0,pred),vmin=min(CLASSES),vmax=max(CLASSES),cmap='tab20');ax[2].set_title('Prithvi + RF (parche16)')
    ax[3].imshow(np.ma.masked_where(y<0,(pred!=y).astype(float)),vmin=0,vmax=1,cmap='Reds');ax[3].set_title('Errores en rojo')
    for a in ax:a.axis('off')
    fig.tight_layout();fig.savefig(OUT/f'02_resultado_{i}.png',dpi=150);plt.show()
rows=[]
for name,mat in totals.items():
    for row in metrics(mat,CLASSES):rows.append({'method':name,**row})
    pd.DataFrame(mat,index=CLASSES,columns=CLASSES).to_csv(OUT/f'confusion_{name}.csv')
byclass=pd.DataFrame(rows);byclass.to_csv(OUT/'metricas_por_clase.csv',index=False)
pd.DataFrame(scene_rows).to_csv(OUT/'metricas_por_escena.csv',index=False)
summary=[]
for name in totals:
    subset=byclass[byclass.method==name]
    if TASK=='cultivos':
        summary.append({'method':name,'f1_macro':subset.f1.mean(),'miou':subset.iou.mean()})
    else:
        pos=subset[subset['class']==1].iloc[0]
        summary.append({'method':name,'f1_positive':pos.f1,'iou_positive':pos.iou,'precision':pos.precision,'recall':pos.recall})
summary=pd.DataFrame(summary);summary.to_csv(OUT/'metricas.csv',index=False)
display(summary);display(byclass)


## 8. Evaluación adicional: media por escena y soporte
El enunciado general exige **media por escena** como agregación principal. La tabla `metricas.csv` original resume matrices de confusión acumuladas (microagregación); esta celda produce una tabla separada de medias por escena, con el mismo conjunto de posiciones válidas. Cuando una métrica sea indefinida, se registra como NaN y se informa cuántas escenas aportan a la media. No ajustar hiperparámetros ni umbrales usando estas etiquetas de prueba.

In [ ]:
# Cálculos derivados únicamente de las salidas reales de la celda anterior.
scene_df = pd.DataFrame(scene_rows)
positive = scene_df[scene_df['class'] == 1].copy()
metric_names = ['iou', 'f1', 'precision', 'recall']
mean_scene = (positive.groupby('method')[metric_names]
              .mean().reset_index())
counts_defined = positive.groupby('method')[metric_names].count().add_prefix('escenas_definidas_').reset_index()
mean_scene = mean_scene.merge(counts_defined, on='method')
mean_scene.to_csv(OUT/'metricas_media_por_escena.csv', index=False)
print('Media por escena (clase positiva):')
display(mean_scene)
print('Distribución de clases válidas de entrenamiento:')
train_distribution = pd.DataFrame({'clase': values, 'pixeles': counts})
train_distribution['fraccion'] = train_distribution.pixeles / train_distribution.pixeles.sum()
train_distribution.to_csv(OUT/'distribucion_train.csv',index=False)
display(train_distribution)
print('Comparación visual — media por escena:')
plot_df = mean_scene.set_index('method')[metric_names]
ax = plot_df.T.plot.bar(figsize=(9,4),rot=0,ylim=(0,1.05),title='Comparación por clase positiva (media por escena)')
ax.set_ylabel('Valor de la métrica'); ax.legend(title='Método')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=2)
ax.text(0.02, 0.92, 'Nota: barras en 0 indican que la clase deslizamiento no fue predicha.', transform=ax.transAxes, fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'03_comparacion_metricas.png',dpi=160);plt.show()


## 8. Trabajo del grupo
1. Explique el problema y la definición de sus clases; identifique el origen de los datos.
2. Justifique por qué este pipeline es **transferencia con encoder congelado**, no un modelo especializado oficial ni fine-tuning completo.
3. Describa y critique la normalización, selección de bandas, recorte, agregación temporal si existe y predicción por parche.
4. Compare modelo y baseline con valores reales; no está garantizado que el modelo gane.
5. Identifique falsos positivos y falsos negativos. Si no existen para una clase/muestra, declárelo sin inventar ejemplos.
6. Discuta clases ausentes y desbalance. ¿Cuánto limita el entrenamiento con cuatro escenas?
7. Proponga una aplicación en Colombia y qué validación adicional necesita.
8. Responda las preguntas de la ficha de su grupo y declare quién ejecutó cada parte.

**Extensiones opcionales:** más escenas fijadas previamente, control con features espectrales + el mismo clasificador, o normalización/preprocesamiento oficial tras verificar compatibilidad. No cambie el método en función de resultados de prueba.


In [ ]:
# Verificación automática de archivos generados (no sustituye la interpretación escrita).
for filename in ['metricas.csv','metricas_por_clase.csv','metricas_por_escena.csv','metricas_media_por_escena.csv','03_comparacion_metricas.png']:
    print(filename, 'OK' if (OUT/filename).exists() else 'FALTA')

## Respuestas y discusión del Grupo 2

**1. Problema y clases.** Buscamos segmentar deslizamientos ya visibles en imágenes satelitales. Clase 0 es fondo o no deslizamiento. Clase 1 es deslizamiento observado. Los datos vienen de Landslide4Sense. No estamos prediciendo un evento futuro ni dibujando un mapa de susceptibilidad.

**2. Por qué es transferencia con encoder congelado.** Descargamos Prithvi-EO-1.0-100M, fijamos una revisión y congelamos todos sus pesos. Solo entrenamos un Random Forest sobre embeddings de parches. No hay fine-tuning del transformer ni uso de un checkpoint oficial especializado en deslizamientos.

**3. Crítica del pipeline.** La normalización usa media y desviación solo de entrenamiento. Se toman seis bandas ópticas de catorce canales; el orden debe confirmarse con el profesor. Las escenas 128×128 se completan a 224×224 con relleno edge y esas posiciones se ignoran en métricas. La predicción por parche 16×16 es gruesa: en este piloto, ningún parche de entrenamiento alcanzó 50 por ciento de píxeles de deslizamiento, así que la clase positiva desapareció antes de entrenar la cabeza.

**4. Comparación real modelo versus línea base.** En la ejecución del 17 de noviembre de 2026, ambos métodos obtuvieron IoU 0, F1 0, precisión 0 y recall 0 para la clase deslizamiento, tanto en agregación global como en media por escena. La línea base constante fue la clase 0, porque esa es la mayoritaria en entrenamiento. El modelo también predijo siempre 0. No superó a la línea base.

**5. Errores observados.** No hubo falsos positivos. Sí hubo falsos negativos: 33 píxeles en la escena de prueba 0 (borde izquierdo, alrededor de la fila 92) y 13 píxeles en la escena de prueba 1 (esquina inferior derecha, alrededor de la fila 121 y columna 125). Las figuras `02_resultado_0.png` y `02_resultado_1.png` muestran esos errores en rojo.

**6. Clases ausentes y desbalance.** A nivel de píxel el entrenamiento sí tiene deslizamientos (919 de 65536 posiciones válidas, cerca de 1,4 por ciento). A nivel de parche, la clase 1 está ausente: 256 parches válidos y todos son clase 0. Con solo cuatro escenas, y con deslizamientos pequeños frente al umbral de mayoría del 50 por ciento, el clasificador no puede aprender la clase positiva.

**7. Aplicación en Colombia.** Proponemos una zona montañosa de Antioquia con inventarios locales, Sentinel-2, elevación y pendiente, separación por evento o región, y evaluación independiente. Esta parte es propuesta; no se ejecutó con datos colombianos.

**Preguntas de la ficha**

Detectar es reconocer un deslizamiento ya visible. Estimar susceptibilidad es decir qué tan propenso es un terreno. Predecir es estimar un evento futuro. Nosotros solo detectamos.

Una carretera o cantera puede confundirse porque muestra suelo descubierto, bordes marcados y tonos parecidos a una cicatriz.

Elevación, pendiente e imágenes anteriores aportarían contexto de relieve y cambio temporal, pero exigiría adaptar y validar el contrato de entrada.

Al transferir entre regiones hay que evitar asumir que el mismo aspecto visual significa lo mismo: cambian vegetación, relieve, clima y sensor. Se necesitan etiquetas locales y pruebas independientes.


In [ ]:
# Resumen automático para el informe: se genera solo después de obtener métricas reales.
if 'summary' not in globals() or 'mean_scene' not in globals():
    raise RuntimeError('Primero ejecuta las celdas de entrenamiento y evaluación.')
print('RESULTADOS GLOBALES (ponderados por píxeles):')
print(summary.to_string(index=False))
print('RESULTADOS MEDIOS POR ESCENA:')
print(mean_scene.to_string(index=False))
print('Los resultados anteriores deben interpretarse junto a las figuras y las limitaciones.')

## Análisis de errores y discusión

Revisamos `02_resultado_0.png`, `02_resultado_1.png` y `escena_*.npz`.

**Caso 1, acierto trivial de fondo.** En ambas escenas de prueba casi todo el área válida es fondo y el modelo la marca como fondo. Eso explica métricas altas de la clase 0, pero no sirve para la tarea principal.

**Caso 2, falso negativo en escena 0.** Hay un deslizamiento pequeño en el borde izquierdo de la zona válida. La referencia lo marca; la predicción no. Son 33 píxeles omitidos. Causa probable: la cabeza nunca vio parches positivos en entrenamiento.

**Caso 3, falso negativo en escena 1.** Hay otra región positiva pequeña en la esquina inferior derecha, con 13 píxeles. Otra vez la predicción queda en cero. No hay falsos positivos en este piloto.

**Conclusión empírica.** Prithvi más Random Forest no supera a la línea base constante en este subconjunto. El cuello de botella no parece ser solo el encoder: la regla de etiqueta mayoritaria por parche elimina la clase rara antes del aprendizaje. Este piloto no debe presentarse como sistema de alerta temprana ni como mapa de susceptibilidad.

### Transferencia a Colombia (propuesta, no validación)
Reunir imágenes compatibles y etiquetas verificadas en regiones montañosas; documentar bandas, resolución y nubes; separar por región o evento; evaluar IoU y F1 de deslizamiento en eventos no vistos; revisar predicciones con experto antes de cualquier uso operativo.


## Consideraciones éticas
Un falso negativo puede ocultar daño real. Un falso positivo puede generar alarmas injustificadas. Por eso no presentamos este resultado como mapa oficial ni como sistema de alerta. Declaramos licencias, procedencia, limitaciones y el uso de asistencia de IA. Cualquier adaptación a Colombia requeriría validación local y revisión experta.


## Conclusiones
En la ejecución del 17 de noviembre de 2026, el modelo y la línea base obtuvieron IoU 0 y F1 0 en la clase deslizamiento. El Random Forest predijo siempre fondo porque la clase positiva desapareció al agregar etiquetas por parche. El piloto sirve para demostrar el flujo y esa limitación metodológica, no para afirmar generalización ni utilidad operativa.


## 9. Exportar resultados y datos de prueba
Se generan dos ZIP: uno de evidencias y otro con el subconjunto real descargado. El ZIP de datos conserva los archivos originales, manifiesto y revisiones; no incluye pesos. Revise licencias antes de redistribuir. Descargue también el notebook con sus salidas mediante Archivo → Descargar → .ipynb.

Colab puede perder los archivos al cerrar la sesión. Guarde copias oportunamente.


In [ ]:
import shutil
shutil.copy2(ROOT/'datos'/'manifest.json', OUT/'manifest_datos.json')
evidence=shutil.make_archive(str(ROOT/'evidencias'),'zip',OUT)
datazip=shutil.make_archive(str(ROOT/'subconjunto_real'),'zip',ROOT/'datos')
print('Evidencias:', evidence)
print('Datos reales:', datazip)
try:
    from google.colab import files
    files.download(evidence)
    files.download(datazip)
except ImportError:
    print('Ejecución local: los ZIP ya quedaron guardados en la carpeta del experimento.')


## Referencias (formato IEEE) y estado de validación
[1] D. Szwarcman et al., “Prithvi-EO-2.0: A versatile multi-temporal foundation model for Earth observation applications,” arXiv:2412.02732, 2024. https://arxiv.org/abs/2412.02732

[2] NASA IMPACT, “HLS Foundation Models,” GitHub. https://github.com/NASA-IMPACT/hls-foundation-os

[3] IBM–NASA Geospatial, “Prithvi-EO-1.0-100M,” Hugging Face. https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-1.0-100M

[4] C. J. Stewart et al., “TerraTorch: The geospatial foundation models toolkit,” arXiv:2503.20563, 2025. https://arxiv.org/abs/2503.20563

[5] Landslide4Sense, Hugging Face Datasets. https://huggingface.co/datasets/sasmit545/Landslide4sense

[6] E. E. Zurek Varela, enunciado general y ficha del Grupo 2, Curso de Inteligencia Artificial, 2026.

[7] Referencia comunitaria mencionada por el docente: https://huggingface.co/srivassid/prithvi-landslide (no oficial).

**Estado:** ejecutado de extremo a extremo en local (CPU, Windows) el 17 de noviembre de 2026. Resultados en `taller_deslizamientos/resultados/` y `entregables/`.


## Lista de entrega
- [x] Modelo, pesos, revisión y datos identificados (`procedencia.json`, `manifest_datos.json`).
- [x] Orden de canales, escala, clases e ignorados documentados.
- [x] Prueba separada de ajustes; no se usaron etiquetas de prueba para entrenar.
- [x] Comparación justa con la línea base y métricas por clase.
- [x] Figuras y análisis vinculados a muestras reales.
- [x] Limitaciones, propuesta regional y riesgos explicados.
- [x] Notebook, informe, CSV, fuentes, integrantes y aportes.
- [ ] Video de 15 minutos basado en diapositivas (pendiente del grupo).
